In [40]:
import pandas as pd
import numpy as np
from data_loading import data_loading, files
casualty_df = data_loading(files['casualty'])
vehicle_df = data_loading(files['vehicle'])
collision_df = data_loading(files['collision'])

#**1. Casualty df, exploring and cleaning**

In [41]:
casualty_df.head()

,collision_index,collision_year,collision_ref_no,vehicle_reference,casualty_reference,casualty_class,sex_of_casualty,age_of_casualty,age_band_of_casualty,casualty_severity,pedestrian_location,pedestrian_movement,car_passenger,bus_or_coach_passenger,pedestrian_road_maintenance_worker,casualty_type,casualty_imd_decile,lsoa_of_casualty,enhanced_casualty_severity,casualty_injury_based,casualty_adjusted_severity_serious,casualty_adjusted_severity_slight,casualty_distance_banding,escooter_flag
0,2022522205219,2022,522205219,2,1,1,1,20,4,2,0,0,0,0,0,-1,4,E01033362,-1,0,1.0,0.0,-1,0
1,2021010288881,2021,010288881,1,1,3,2,7,2,2,9,6,0,0,0,0,1,E01004340,-1,0,1.0,0.0,-1,0
2,2021010288939,2021,010288939,1,1,3,1,21,5,3,8,4,0,0,0,0,3,E01001823,-1,0,0.0,1.0,-1,0
3,2021010289554,2021,010289554,1,1,3,2,11,3,3,5,1,0,0,0,0,2,E01000931,-1,0,0.0,1.0,-1,0
4,2021010290262,2021,010290262,1,1,3,2,31,6,2,5,1,0,0,0,0,6,E01003226,-1,0,1.0,0.0,-1,0


In [42]:
print(casualty_df.columns)

Index(['collision_index', 'collision_year', 'collision_ref_no',
       'vehicle_reference', 'casualty_reference', 'casualty_class',
       'sex_of_casualty', 'age_of_casualty', 'age_band_of_casualty',
       'casualty_severity', 'pedestrian_location', 'pedestrian_movement',
       'car_passenger', 'bus_or_coach_passenger',
       'pedestrian_road_maintenance_worker', 'casualty_type',
       'casualty_imd_decile', 'lsoa_of_casualty', 'enhanced_casualty_severity',
       'casualty_injury_based', 'casualty_adjusted_severity_serious',
       'casualty_adjusted_severity_slight', 'casualty_distance_banding',
       'escooter_flag'],
      dtype='object')


In [43]:
casualty_raw = casualty_df.copy()
casualty_cols = [
    'collision_index', 'casualty_reference', 'casualty_class',
    'sex_of_casualty',  'age_of_casualty', 'casualty_type', 'pedestrian_movement', 'car_passenger', 'bus_or_coach_passenger',
    'pedestrian_road_maintenance_worker', 'casualty_imd_decile', 'escooter_flag'
]
casualty_df = casualty_df[casualty_cols]

###**Meaning of each selected column**


**collision_index:** merge key

**casualty_reference:** for unique casualty count

**casualty_class**: role of casualty

**sex_of_casualty:** Gender distribution

**age_of_casualty:**	Age analysis

**casualty_type:**	Casualty type

**pedestrian_location:**	Pedestrian-related information

**pedestrian_movement:**	Pedestrian movement

**car_passenger:**	Car passenger information

**bus_or_coach_passenger:**	Bus passenger information

**pedestrian_road_maintenance_worker:**	Road maintenance worker status

**casualty_imd_decile:**	Socioeconomic indicator

**escooter_flag:**	E-scooter involvement

In [44]:
# Rows and columns:
print(f"Rows: {casualty_df.shape[0]} \nColumns: {casualty_df.shape[1]}")

Rows: 652821 
Columns: 12


In [45]:
# Information:
casualty_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 652821 entries, 0 to 652820
Data columns (total 12 columns):
 #   Column                              Non-Null Count   Dtype 
---  ------                              --------------   ----- 
 0   collision_index                     652821 non-null  object
 1   casualty_reference                  652821 non-null  int64 
 2   casualty_class                      652821 non-null  int64 
 3   sex_of_casualty                     652821 non-null  int64 
 4   age_of_casualty                     652821 non-null  int64 
 5   casualty_type                       652821 non-null  int64 
 6   pedestrian_movement                 652821 non-null  int64 
 7   car_passenger                       652821 non-null  int64 
 8   bus_or_coach_passenger              652821 non-null  int64 
 9   pedestrian_road_maintenance_worker  652821 non-null  int64 
 10  casualty_imd_decile                 652821 non-null  int64 
 11  escooter_flag                       652

In [46]:
(casualty_df['age_of_casualty'] == -1).value_counts()

,count
age_of_casualty,
False,638715
True,14106


In [49]:
casualty_mapping = {
    'casualty_class': {
        1: 'driver or rider',
        2: 'passeger',
        3: 'pedestrian'
    },

    'sex_of_casualty': {
        1: 'male',
        2: 'female',
        9: 'unknown',
        -1: 'data missing or out of range'
    },

    'casualty_type': {
        0: 'pedestrian',
        1: 'cyclist',
        2: 'motorcycle 50cc and under rider or passenger',
        3: 'motorcycle 125cc and under rider or passenger',
        4: 'motorcycle over 125cc and upto 500cc rider or passenger',
        5: 'motorcycle over 500cc rider or passenger',
        8: 'taxi or private hire car occupant',
        9: 'car occupant',
        10: 'minibus (8 - 16 seats) occupant',
        11: 'bus or coach occupant (17 or more pass seats)',
        16: 'horse rider',
        17: 'agricultural vehicle occupant',
        18: 'tram occupant',
        19: 'van or goods vehicle (3.5 tonnes mgw or under) occupant',
        20: 'goods vehicle (over 3.5t. and under 7.5t.) occupant',
        21: 'goods vehicle (7.5 tonnes mgw and over) occupant',
        22: 'mobility scooter rider',
        23: 'electric motorcycle rider or passenger',
        33: 'personal powered transporter',
        90: 'other vehicle occupant',
        97: 'motorcycle - unknown cc rider or passender',
        98: 'goods vehicle (unknown weight) occupant',
        99: 'unknown vehicle type (self reported only)',
        -1: 'data missing or out of range'
    },

    'pedestrian_movement': {
        0: 'not a pedestrian',
        1: "crossing from driver's nearside",
        2: 'crossing from nearside - masked by parked or stationary vehicle',
        3: "crossing from drivers offside",
        4: 'Crossing from offside - masked by  parked or stationary vehicle',
        5: 'In carriageway, stationary - not crossing  (standing or playing)',
        6: 'In carriageway, stationary - not crossing  (standing or playing) - masked by parked or stationary vehicle',
        7: 'walking along in carriagway, facing traffic',
        8: 'walking along in carriagway, back to traffic',
        9: 'unknown or other',
        -1: 'data missing or out of range'
    },

    'car_passenger': {
        0: 'not car passenger',
        1: 'front seat passenger',
        2: 'rear seat passenger',
        9: 'unknown (self reported)',
        -1: 'data missing or out of range'
    },

    'bus_or_coach_passenger': {
        0: 'not a bus or coach passenger',
        1: 'boarding',
        2: 'alighting',
        3: 'standing passenger',
        4: 'seated passenger',
        9: 'unknown (self reported)',
        -1: 'data missing or out of range'
    },

    'pedestrian_road_maintenance_worker': {
        0: 'no or not applicable',
        1: 'yes',
        2: 'not known',
        -1: 'data missing or out of range'
    },

    'casualty_imd_decile': {
        1: 'most deprived 10%',
        2: 'most deprived 10-20%',
        3: 'most deprived 20-30%',
        4: 'most deprived 30-40%',
        5: 'most deprived 40-50%',
        6: 'less deprived 40-50%',
        7: 'less deprived 30-40%',
        8: 'less deprived 20-30%',
        9: 'less deprived 10-20%',
        10: 'least deprived 10%',
        -1: 'data missing or out of range'
    },

    'escooter_flag': {
        0: 'casualty was using an e-scooter',
        1: 'casualty was not using an e-scooter'
    }
}

for col,mapping in casualty_mapping.items():
  casualty_df[f'{col}_label'] = casualty_df[col].map(mapping)

casualty_df['age_of_casualty_label'] = casualty_df['age_of_casualty'].replace({
    -1: "data missing or out of range"
})

In [50]:
casualty_df.head(20)

,collision_index,casualty_reference,casualty_class,sex_of_casualty,age_of_casualty,casualty_type,pedestrian_movement,car_passenger,bus_or_coach_passenger,pedestrian_road_maintenance_worker,casualty_imd_decile,escooter_flag,casualty_class_label,sex_of_casualty_label,casualty_type_label,pedestrian_movement_label,car_passenger_label,bus_or_coach_passenger_label,pedestrian_road_maintenance_worker_label,casualty_imd_decile_label,escooter_flag_label,age_of_casualty_label
0,2022522205219,1,1,1,20,-1,0,0,0,0,4,0,driver or rider,male,data missing or out of range,not a pedestrian,not car passenger,not a bus or coach passenger,no or not applicable,most deprived 30-40%,casualty was using an e-scooter,20
1,2021010288881,1,3,2,7,0,6,0,0,0,1,0,pedestrian,female,pedestrian,"In carriageway, stationary - not crossing (st...",not car passenger,not a bus or coach passenger,no or not applicable,most deprived 10%,casualty was using an e-scooter,7
2,2021010288939,1,3,1,21,0,4,0,0,0,3,0,pedestrian,male,pedestrian,Crossing from offside - masked by parked or s...,not car passenger,not a bus or coach passenger,no or not applicable,most deprived 20-30%,casualty was using an e-scooter,21
3,2021010289554,1,3,2,11,0,1,0,0,0,2,0,pedestrian,female,pedestrian,crossing from driver's nearside,not car passenger,not a bus or coach passenger,no or not applicable,most deprived 10-20%,casualty was using an e-scooter,11
4,2021010290262,1,3,2,31,0,1,0,0,0,6,0,pedestrian,female,pedestrian,crossing from driver's nearside,not car passenger,not a bus or coach passenger,no or not applicable,less deprived 40-50%,casualty was using an e-scooter,31
5,2021010291846,1,3,2,24,0,1,0,0,0,2,0,pedestrian,female,pedestrian,crossing from driver's nearside,not car passenger,not a bus or coach passenger,no or not applicable,most deprived 10-20%,casualty was using an e-scooter,24
6,2021010299301,1,3,1,48,0,1,0,0,0,-1,0,pedestrian,male,pedestrian,crossing from driver's nearside,not car passenger,not a bus or coach passenger,no or not applicable,data missing or out of range,casualty was using an e-scooter,48
7,2021010308635,1,3,2,18,0,1,0,0,0,7,0,pedestrian,female,pedestrian,crossing from driver's nearside,not car passenger,not a bus or coach passenger,no or not applicable,less deprived 30-40%,casualty was using an e-scooter,18
8,2021010314426,1,3,1,12,0,3,0,0,0,3,0,pedestrian,male,pedestrian,crossing from drivers offside,not car passenger,not a bus or coach passenger,no or not applicable,most deprived 20-30%,casualty was using an e-scooter,12
9,2021010316116,1,3,2,23,0,2,0,0,0,1,0,pedestrian,female,pedestrian,crossing from nearside - masked by parked or s...,not car passenger,not a bus or coach passenger,no or not applicable,most deprived 10%,casualty was using an e-scooter,23


In [51]:
casualty_df.isnull().sum()

,0
collision_index,0
casualty_reference,0
casualty_class,0
sex_of_casualty,0
age_of_casualty,0
casualty_type,0
pedestrian_movement,0
car_passenger,0
bus_or_coach_passenger,0
pedestrian_road_maintenance_worker,0
